# 04b - Vector Database RAG Experiment

This notebook experiments with vector database retrieval for the CardioBot RAG system.

Pipeline:
1. Load raw cardiovascular documents
2. Split documents into chunks
3. Convert chunks into sentence embeddings
4. Store embeddings in FAISS vector index
5. Retrieve top-k contexts using semantic similarity
6. Generate answers using Qwen2.5 + LoRA

This notebook is experimental and does not overwrite the original TF-IDF RAG results.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

In [2]:
BASE_DIR = Path.cwd()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = BASE_DIR / "results"
MODEL_DIR = BASE_DIR / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_PATH = MODEL_DIR / "qwen2_5_1_5b_cardio_lora"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Base directory:", BASE_DIR)
print("Raw directory:", RAW_DIR)
print("Results directory:", RESULTS_DIR)
print("LoRA path:", LORA_PATH)
print("Device:", device)

Base directory: d:\CardioBot_NLP_Final
Raw directory: d:\CardioBot_NLP_Final\data\raw
Results directory: d:\CardioBot_NLP_Final\results
LoRA path: d:\CardioBot_NLP_Final\models\qwen2_5_1_5b_cardio_lora
Device: cuda


In [3]:
raw_files = sorted(RAW_DIR.glob("*.txt"))

documents = []

for file_path in raw_files:
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    
    documents.append({
        "source": file_path.name,
        "text": text
    })

print("Total raw documents:", len(documents))

for doc in documents[:10]:
    print("-", doc["source"])

Total raw documents: 29
- Angiography.txt
- Angioplasty and stent.txt
- Arrhythmia.txt
- Atherosclerosis.txt
- Blood pressure measurement.txt
- Blood_Flow.txt
- Cardiac ablation.txt
- Cardiac rehabilitation.txt
- Cardiomyopathy.txt
- Cholesterol.txt


In [4]:
def clean_text(text):
    text = text.replace("\r", "\n")
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def chunk_text(documents, chunk_size=180, overlap=30):
    chunks = []

    for doc in documents:
        text = clean_text(doc["text"])
        paragraphs = [p.strip() for p in text.split("\n") if p.strip()]

        for para in paragraphs:
            if len(para.split()) < 20:
                continue

            sentences = re.split(r'(?<=[.!?])\s+', para)

            current_chunk = []
            current_length = 0

            for sentence in sentences:
                words = sentence.split()

                if current_length + len(words) <= chunk_size:
                    current_chunk.append(sentence)
                    current_length += len(words)
                else:
                    chunk_str = " ".join(current_chunk).strip()

                    if len(chunk_str.split()) >= 30:
                        chunks.append({
                            "source": doc["source"],
                            "text": chunk_str
                        })

                    current_chunk = current_chunk[-1:]
                    current_length = sum(len(s.split()) for s in current_chunk)

                    current_chunk.append(sentence)
                    current_length += len(words)

            if current_chunk:
                chunk_str = " ".join(current_chunk).strip()

                if len(chunk_str.split()) >= 30:
                    chunks.append({
                        "source": doc["source"],
                        "text": chunk_str
                    })

    # remove duplicate chunks
    unique_chunks = []
    seen = set()

    for chunk in chunks:
        if chunk["text"] not in seen:
            unique_chunks.append(chunk)
            seen.add(chunk["text"])

    return unique_chunks


chunks = chunk_text(documents)

print("Total chunks:", len(chunks))
pd.DataFrame(chunks).head()

Total chunks: 303


,source,text
0,Angiography.txt,Angiography is a type of X-ray imaging used to...
1,Angiography.txt,Angiography is used to evaluate the condition ...
2,Angiography.txt,Angiography is usually performed in a hospital...
3,Angiography.txt,Angiography is generally considered safe and p...
4,Angiography.txt,There are several types of angiography dependi...


In [5]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print("Embedding model loaded:", EMBEDDING_MODEL)

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


In [6]:
chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", chunk_embeddings.shape)
print("Number of chunks:", len(chunks))

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embedding shape: (303, 384)
Number of chunks: 303


In [7]:
embedding_dim = chunk_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(chunk_embeddings.astype("float32"))

print("FAISS index created.")
print("Total vectors in index:", faiss_index.ntotal)

FAISS index created.
Total vectors in index: 303


In [8]:
def retrieve_faiss_context(question, top_k=3):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "source": chunks[idx]["source"],
            "text": chunks[idx]["text"],
            "score": float(score)
        })

    return results

In [9]:
test_questions = [
    "How does blood flow through the heart and body?",
    "What test can check if my heart rhythm is irregular?",
    "Can high cholesterol cause heart problems even if I feel healthy?",
    "What are the warning signs of a stroke?"
]

for question in test_questions:
    retrieved = retrieve_faiss_context(question, top_k=3)

    print("=" * 100)
    print("Question:", question)
    
    for i, item in enumerate(retrieved, start=1):
        print(f"\nRank {i}")
        print("Source:", item["source"])
        print("Score:", round(item["score"], 4))
        print("Text:", item["text"][:500])

Question: How does blood flow through the heart and body?

Rank 1
Source: Blood_Flow.txt
Score: 0.7118
Text: Blood flows through the heart and body in a continuous cycle. Oxygen-poor blood enters the right atrium through the superior and inferior vena cava and moves into the right ventricle through the tricuspid valve. The right ventricle then pumps this blood to the lungs through the pulmonary artery, where it becomes oxygenated.

Rank 2
Source: circulation.txt
Score: 0.7079
Text: Systemic circulation carries oxygenated blood from the heart to the rest of the body. Blood is pumped from the left ventricle into the aorta and then flows through arteries, arterioles, and capillaries. At the capillary level, oxygen and nutrients are delivered to tissues, and carbon dioxide and waste products are collected. The deoxygenated blood then returns to the heart through veins and enters the right atrium.

Rank 3
Source: circulation.txt
Score: 0.6872
Text: Blood flow occurs in a continuous cycle. D

In [10]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

torch_dtype = torch.float16 if device == "cuda" else torch.float32

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch_dtype,
    trust_remote_code=True,
    device_map="auto" if device == "cuda" else None
)

model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

print("Qwen LoRA model loaded.")

Qwen LoRA model loaded.


In [11]:
SYSTEM_PROMPT = (
    "You are CardioBot, a cardiovascular health education assistant. "
    "Answer only using the provided context. "
    "If the answer is not available in the context, say that the information is not available in the knowledge base. "
    "Do not provide diagnosis, prescriptions, or emergency medical decisions. "
    "For emergency symptoms, advise the user to seek immediate medical help."
)


def build_vector_rag_prompt(question, retrieved_contexts):
    context_text = ""

    for i, item in enumerate(retrieved_contexts, start=1):
        context_text += f"[Context {i} | Source: {item['source']}]\n"
        context_text += item["text"] + "\n\n"

    user_prompt = (
        f"Context:\n{context_text}\n"
        f"Question: {question}\n\n"
        f"Answer clearly and completely based only on the context. "
        f"If the question asks for a process, pathway, or flow, explain the full sequence step by step from beginning to end. "
        f"Do not skip important steps if they are present in the context."
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [12]:
def safety_check(question):
    q = question.lower()

    prescription_keywords = [
        "prescribe", "medicine should i take", "what medicine",
        "best medicine", "dosage", "dose", "stop taking",
        "should i stop", "can i stop", "medication"
    ]

    diagnosis_keywords = [
        "diagnose", "do i have", "am i having",
        "whether i have", "confirm if i have"
    ]

    emergency_keywords = [
        "severe chest pain", "sudden chest pain",
        "can't breathe", "cannot breathe",
        "fainting", "face drooping",
        "trouble speaking", "stroke symptoms",
        "heart attack symptoms"
    ]

    if any(keyword in q for keyword in emergency_keywords):
        return (
            "This may be an emergency symptom. I cannot diagnose your condition, "
            "but you should seek immediate medical help or contact local emergency services right away."
        )

    if any(keyword in q for keyword in prescription_keywords):
        return (
            "I cannot prescribe, recommend, change, or stop medication. "
            "Please consult a licensed healthcare professional for medication advice, especially for chest pain or heart-related symptoms."
        )

    if any(keyword in q for keyword in diagnosis_keywords):
        return (
            "I cannot diagnose whether you have a specific condition. "
            "I can explain general cardiovascular information, but diagnosis requires evaluation by a healthcare professional."
        )

    return None

In [13]:
def generate_vector_rag_answer(
    question,
    top_k=3,
    max_new_tokens=300,
    min_score=0.25
):
    safety_response = safety_check(question)
    if safety_response is not None:
        return {
            "answer": safety_response,
            "retrieved_contexts": []
        }

    retrieved = retrieve_faiss_context(question, top_k=top_k)

    if retrieved[0]["score"] < min_score:
        return {
            "answer": "The information is not available in the cardiovascular knowledge base.",
            "retrieved_contexts": retrieved
        }

    prompt = build_vector_rag_prompt(question, retrieved)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.15,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()

    return {
        "answer": answer,
        "retrieved_contexts": retrieved
    }

In [14]:
def ask_vector_rag(question):
    result = generate_vector_rag_answer(question)

    print("Question:")
    print(question)

    print("\nAnswer:")
    print(result["answer"])

    if result["retrieved_contexts"]:
        print("\nRetrieved Sources:")
        for item in result["retrieved_contexts"]:
            print(f"- {item['source']} | score: {item['score']:.4f}")


demo_questions = [
    "How does blood flow through the heart and body?",
    "What test can check if my heart rhythm is irregular? ECG or Holter Monitor? What are the differences?",
    "Can high cholesterol cause heart problems even if I feel healthy?",
    "What are the warning signs of a stroke?",
    "Can you prescribe medicine for my chest pain?"
]

for q in demo_questions:
    print("=" * 100)
    ask_vector_rag(q)

Question:
How does blood flow through the heart and body?

Answer:
Blood flows through the heart and body in a continuous cycle. Oxygen-poor blood enters the right atrium through the superior and inferior vena cava and moves into the right ventricle through the tricuspid valve. The right ventricle then pumps this blood to the lungs through the pulmonary artery, where it becomes oxygenated. Oxygenated blood returns to the heart through the pulmonary veins and enters the left atrium. From there, it flows into the left ventricle and is pumped out to the rest of the body via the aorta and other arteries.

Retrieved Sources:
- Blood_Flow.txt | score: 0.7118
- circulation.txt | score: 0.7079
- circulation.txt | score: 0.6872
Question:
What test can check if my heart rhythm is irregular? ECG or Holter Monitor? What are the differences?

Answer:
An ECG or Holter monitor can check if your heart rhythm is irregular. An ECG is a quick test that uses sensors placed on the chest to measure heart rh

In [15]:
def read_jsonl(path):
    data = []
    
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            if line:
                data.append(json.loads(line))
    
    return data


test_data = read_jsonl(PROCESSED_DIR / "test.jsonl")

print("Test size:", len(test_data))
pd.DataFrame(test_data).head()

Test size: 34


,id,topic,source,question,answer
0,qa_047,Cardiomyopathy,Cardiomyopathy.txt,What is dilated cardiomyopathy?,Dilated cardiomyopathy is a type of cardiomyop...
1,qa_057,Heart Valve Disease,Heart Valve Disease.txt,What is valve regurgitation?,Valve regurgitation happens when valve flaps d...
2,qa_099,Blood Flow,Blood_Flow.txt,What are the main functions of blood flow?,Blood flow delivers oxygen and nutrients to or...
3,qa_135,Heart Failure,heart_failure.txt,What are the ACC/AHA stages of heart failure?,The ACC/AHA stages of heart failure are Stage ...
4,qa_070,Stroke,Stroke.txt,How is hemorrhagic stroke treated?,Hemorrhagic stroke treatment focuses on contro...


In [16]:
faiss_rag_results = []

for i, item in enumerate(test_data, start=1):
    question = item["question"]
    
    result = generate_vector_rag_answer(
        question=question,
        top_k=3,
        max_new_tokens=300,
        min_score=0.25
    )

    retrieved_contexts = result["retrieved_contexts"]

    top_source = retrieved_contexts[0]["source"] if retrieved_contexts else None
    top_score = retrieved_contexts[0]["score"] if retrieved_contexts else None

    faiss_rag_results.append({
        "id": item["id"],
        "topic": item["topic"],
        "source": item["source"],
        "question": question,
        "reference_answer": item["answer"],
        "faiss_rag_answer": result["answer"],
        "top_source": top_source,
        "top_score": top_score,
        "retrieved_contexts": retrieved_contexts
    })

    print(f"[{i}/{len(test_data)}] {question}")
    print(result["answer"][:250])
    print("-" * 80)

[1/34] What is dilated cardiomyopathy?
Dilated cardiomyopathy occurs when the heart's chambers become thinner and larger, usually starting in the left ventricle. This makes it harder for the heart to pump blood effectively.
--------------------------------------------------------------------------------
[2/34] What is valve regurgitation?
Valve regurgitation occurs when the valve flaps do not close properly after opening. This allows some blood to flow backward into the previous chamber instead of flowing forward to the next chamber.
--------------------------------------------------------------------------------
[3/34] What are the main functions of blood flow?
The main functions of blood flow include delivering oxygen and nutrients to organs and tissues, removing carbon dioxide and metabolic waste, transporting immune cells, and maintaining blood pressure.
--------------------------------------------------------------------------------
[4/34] What are the ACC/AHA stages of heart fail

In [17]:
faiss_rag_output_path = RESULTS_DIR / "faiss_rag_answers.json"
faiss_rag_csv_path = RESULTS_DIR / "faiss_rag_results.csv"

with open(faiss_rag_output_path, "w", encoding="utf-8") as f:
    json.dump(faiss_rag_results, f, indent=2, ensure_ascii=False)

pd.DataFrame(faiss_rag_results).to_csv(faiss_rag_csv_path, index=False)

print("Saved FAISS RAG results to:")
print(faiss_rag_output_path)
print(faiss_rag_csv_path)

Saved FAISS RAG results to:
d:\CardioBot_NLP_Final\results\faiss_rag_answers.json
d:\CardioBot_NLP_Final\results\faiss_rag_results.csv


In [18]:
faiss_rag_df = pd.DataFrame(faiss_rag_results)

faiss_rag_df["source_match"] = faiss_rag_df["source"] == faiss_rag_df["top_source"]


def top3_source_match(row):
    if not row["retrieved_contexts"]:
        return False
    
    retrieved_sources = [ctx["source"] for ctx in row["retrieved_contexts"]]
    return row["source"] in retrieved_sources


faiss_rag_df["top3_source_match"] = faiss_rag_df.apply(top3_source_match, axis=1)

faiss_rag_summary = {
    "model": "FAISS RAG + Qwen LoRA",
    "test_size": len(faiss_rag_df),
    "source_match_accuracy": round(float(faiss_rag_df["source_match"].mean()), 4),
    "top3_source_match_accuracy": round(float(faiss_rag_df["top3_source_match"].mean()), 4),
    "average_top_score": round(float(faiss_rag_df["top_score"].mean()), 4)
}

faiss_rag_summary

{'model': 'FAISS RAG + Qwen LoRA',
 'test_size': 34,
 'source_match_accuracy': 0.7353,
 'top3_source_match_accuracy': 0.9706,
 'average_top_score': 0.6953}

In [19]:
with open(RESULTS_DIR / "faiss_rag_summary.json", "w", encoding="utf-8") as f:
    json.dump(faiss_rag_summary, f, indent=2, ensure_ascii=False)

print("Saved FAISS RAG summary.")

Saved FAISS RAG summary.
